# Phase C v2 — Training (10-channel, 7-class UNet)
Forked from the v1 `phase_c_train_colab_v3.ipynb`. The only structural change is
the input: **7 one-hot classes** (glass un-fold) + tx_blob + freq + dist =
**IN_CH = 10** (v1 was 9). Everything else — masked MSE+grad loss, Drive-resume,
baselines, ONNX parity gate — is unchanged.

**Runtime → GPU.** Requires the v2 dataset from `phase_b_v2_dataset.ipynb`.

In [ ]:
import os
if not os.path.exists('indoor-walk-test'):
    os.system('git clone --depth 1 https://github.com/cgm2179/indoor-walk-test.git')
%cd indoor-walk-test
import torch, json, numpy as np
manifest = json.load(open('SIM/manifest.json'))
NORM = manifest['norm']; PL_LO, PL_RNG = NORM['pl_min_db'], NORM['pl_range_db']
H, W = manifest['grid_shape']; SIGMA = manifest['tx_blob_sigma_cells']
CELL = manifest['cell_size_m']; DN = manifest.get('dist_channel_norm', 3.0)
IN_CH = 10   # v2: 7 classes + tx + freq + dist  (v1 was 9)
N_CLASSES = 7
print('GPU:', torch.cuda.is_available(), '| IN_CH', IN_CH, '| classes', N_CLASSES)

## Prerequisite & what changes vs v3
- Dataset must be the **v2** set (Drive folder `indoor-walk-test-dataset-v2` or
  similar) generated by Phase B v2 with the 7-class grid + 9-band union.
- `build_input` one-hot loop runs over **7** classes, not 6.
- Freq normalization spans **619–6125 MHz** (the union), not 2400–6125.
- Baselines, resume, and ONNX export are copied verbatim from v3 — see that
  notebook; only IN_CH / N_CLASSES / the freq range differ.

In [ ]:
# v2 input builder — the ONE function that differs from v3 (7-class one-hot).
# The SAME logic must be mirrored in simulator_tab.js when the v2 model ships.
import numpy as np, torch
yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
def build_input_v2(grid7, tx_x, tx_y, ff):
    x = np.empty((IN_CH, H, W), np.float32)
    for c in range(N_CLASSES):
        x[c] = (grid7 == c)
    d = np.hypot(xx - tx_x, yy - tx_y)
    x[N_CLASSES]     = np.exp(-d**2 / (2*SIGMA**2))   # tx blob
    x[N_CLASSES + 1] = ff                              # freq feature
    x[N_CLASSES + 2] = np.log10(np.maximum(d*CELL, 1.0)) / DN
    return x
print('build_input_v2 -> tensor shape', (IN_CH, H, W))

In [ ]:
# The UNet, training loop, baselines, resume, and ONNX-export cells are
# identical to SIM/phase_c_train_colab_v3.ipynb EXCEPT:
#   UNet(in_ch=IN_CH)          # 10, not 9
#   build_input -> build_input_v2  (7-class one-hot)
#   freq_log_lo = 619          # union lower bound
# Copy those cells from v3 and apply these three edits. Keeping the diff this
# small is deliberate (DECISIONS D2: only the input/grid change between physics
# generations; the learner is unchanged).
print('copy v3 model/train/eval/onnx cells; apply the 3 edits above')